# Checkpointed scientific experiment

This notebook demonstrates the experiment runner on two deliberately small cells. A scientific run should replace these with the frozen TESS grid and increase the simulation counts.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    SyntheticExperimentPlan,
    ValidationCase,
    make_observing_window,
    run_synthetic_experiment,
)

In [ ]:
def make_case(name, split, numax, delta_nu, amplitude, random_missing, seed):
    window = make_observing_window(
        duration_days=0.8,
        cadence_seconds=120.0,
        gaps_days=((0.39, 0.41),),
        random_missing_fraction=random_missing,
        seed=seed,
    )
    width = 400.0
    return ValidationCase(
        name=name,
        split=split,
        window=window,
        simulation=SimulationConfig(
            white_noise_sigma=0.2,
            granulation_amplitude=0.1,
            numax_uhz=numax,
            delta_nu_uhz=delta_nu,
            envelope_width_uhz=width,
            oscillation_amplitude=amplitude,
        ),
        published_centres_uhz=np.linspace(0.5 * numax, 1.5 * numax, 11),
        restricted_centres_uhz=np.linspace(numax - 0.5 * width, numax + 0.5 * width, 7),
        filter_width_uhz=500.0,
        delta_nu_grid_uhz=np.linspace(0.9 * delta_nu, 1.1 * delta_nu, 5),
        segments_days=((0.0, 0.4), (0.4, 0.8)),
        coherent_contaminants={
            "single_line": CoherentSignalConfig(numax, 0.8),
            "harmonic_comb": CoherentSignalConfig(numax / 3.0, 0.6, harmonics=3),
        },
        segment_systematics={
            "variance_jump": [SegmentSystematicConfig(0.4, 0.8, amplitude_scale=4.0)],
        },
        max_lag_seconds=35_000.0,
    )


cases = (
    make_case("train_mid_numax", "train", 1000.0, 100.0, 1.8, 0.01, 1),
    make_case("validation_sparse_window", "validation", 800.0, 80.0, 1.4, 0.10, 2),
)

In [ ]:
plan = SyntheticExperimentPlan(
    name="tutorial-grid",
    cases=cases,
    calibration_realizations=16,
    evaluation_realizations=4,
    target_false_positive_rate=0.25,
    reliability_bins=4,
    seed=42,
)
manifest = plan.to_manifest()
manifest["fingerprint"], [case["name"] for case in manifest["cases"]]

In [ ]:
with TemporaryDirectory() as directory:
    run = run_synthetic_experiment(plan, directory, workers=1)
    metric_lines = Path(run.metrics_path).read_text().splitlines()
    reliability_lines = Path(run.reliability_path).read_text().splitlines()
    resumed = run_synthetic_experiment(plan, directory, workers=1)

len(metric_lines) - 1, len(reliability_lines) - 1, resumed.reused_cases

For the real run, use a persistent output directory and a process count appropriate to the machine. Each case checkpoint is reusable, so a stopped run can be submitted again with `resume=True`.